# Exercise 2 - How many HUXt runs does the surrogate actually need?

**Time budget:** ~30 min &nbsp;|&nbsp; **Reference:** notebook 04, Task 3 &nbsp;|&nbsp; **No *new* HUXt runs** (reuses your Exercise-1 event batch)

Exercise 1 is busy running 300 samples - but is 300 necessary? Train the surrogate on growing
subsets of your event's batch and watch its skill and its *honesty* (does the GP's
own uncertainty match the real error?) as a function of the number of runs.

You are given a `fit_surrogate(train_df)` helper (the Task-3 recipe) and a **fixed held-out test
set**, so every training size is scored on the same rows. The experiment itself is yours to write.

**By the end you can:** plot surrogate error and uncertainty vs training-set size, and decide how
many HUXt runs a new event is worth.

In [ ]:
# --- Google Colab bootstrap (no-op locally) ---
import sys, os

if "google.colab" in sys.modules:
    REPO_URL = os.environ.get("CONECAST_REPO", "https://github.com/georgemilosh/conecast")
    REPO_DIR = "/content/conecast"
    if not os.path.isdir(REPO_DIR):
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
    try:
        import sunpy, huxt, wsaplus  # noqa: F401
        print("Colab bootstrap complete; cwd =", os.getcwd())
    except ModuleNotFoundError:
        print("Installing sunpy + WSA+ + HUXt (one-time, ~2 min)...")
        os.system("pip install -q sunpy wsaplus")
        os.system("pip install -q "
                  "'huxt @ git+https://github.com/University-of-Reading-Space-Science/HUXt'")
        print("Done - restarting the runtime. When it reconnects, RUN THIS CELL AGAIN.")
        os.kill(os.getpid(), 9)
else:
    print("Not in Colab - using the local checkout.")

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
BASE_DIR = cwd if (cwd / "scripts").exists() else cwd.parent
SCRIPT_DIR = BASE_DIR / "scripts"
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))
print("BASE_DIR =", BASE_DIR)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessClassifier, GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error
import gp_huxt_surrogate as gp

PARAM_NAMES = gp.PARAM_NAMES
# SIZES is derived from the actual batch size below (after the train/test split).

DATA_ROOT = BASE_DIR / "data_dir" / "sw"
GP_ROOT   = BASE_DIR / "runs" / "gp_surrogate"

EVENT = ...   # TODO: the event id you provisioned in Exercise 1 (an id from data_dir/events.csv)

# Exercises 2-4 reuse the single event you set up in Exercise 1 (we ship no runs/ data). If its
# batch isn't here yet, generate it (~7-8 min, ~1.5 s/run); its boundary must already exist -
# i.e. Exercise 1 must have been run for this EVENT.
res_path = GP_ROOT / EVENT / "results.csv"
boundary = DATA_ROOT / EVENT / f"v_boundary_{EVENT}.npz"
if not res_path.exists():
    if not boundary.exists():
        raise FileNotFoundError(
            f"No batch or boundary for {EVENT!r} - run Exercise 1 first to provision this event."
        )
    print(f"Boundary present but no batch yet - running 300 HUXt samples for {EVENT} (~7-8 min)...")
    _span = dict(inject_hour=1.0, longitude=30.0, latitude=20.0, width=40.0, speed_fraction=0.25)
    if not (GP_ROOT / EVENT / "design.csv").exists():
        gp.make_design(EVENT, DATA_ROOT, GP_ROOT, n=300, seed=42, force=False,
                       span_inject_hour=_span["inject_hour"], span_longitude=_span["longitude"],
                       span_latitude=_span["latitude"], span_width=_span["width"],
                       span_speed_fraction=_span["speed_fraction"])
    gp.run_design(EVENT, DATA_ROOT, GP_ROOT, limit=300, detector_threshold=0.25,
                  detector_lag=2, smooth_window=5, detector_method="enhancement",
                  baseline_window=24, rerun_completed=False)

res = pd.read_csv(res_path)
res = res.loc[res["status"] == "completed"].copy()
res["hit"] = res["hit"].map(lambda v: str(v).strip().lower() in {"1", "true", "yes", "y"})

# A fixed held-out TEST set (rows never used for training) so every N is scored on the same rows.
# Expects your Exercise-1 batch (>= 300 runs). Hold out a fixed test set (last
# ~20%) so every training size is scored on the same rows; train on growing subsets of the rest.
n_test     = max(50, len(res) // 6)
train_pool = res.iloc[:-n_test].reset_index(drop=True)
test       = res.iloc[-n_test:].reset_index(drop=True)
SIZES      = sorted({n for n in [25, 50, 100, 200, len(train_pool)] if n <= len(train_pool)})
test_hit_true = test["hit"].astype(int).to_numpy()                       # classifier target
test_hits = test.loc[test["hit"] & np.isfinite(test["arrival_time_hr"])] # rows with a true arrival
X_test_hits = test_hits[PARAM_NAMES].to_numpy(float)
y_test_hits = test_hits["arrival_time_hr"].to_numpy(float)

ctx = gp.load_huxt_context(EVENT, DATA_ROOT)
theta0 = ctx["theta0"]
print(f"{len(res)} rows -> train pool {len(train_pool)}, fixed test {len(test)} ({len(test_hits)} hits)")

def fit_surrogate(train_df):
    """Fit hit classifier + arrival regressor on train_df; return predict_hit, predict_arrival.
    Same recipe as tutorial notebook 04 Task 3 - given so you can focus on the experiment."""
    x = train_df[PARAM_NAMES].to_numpy(float)
    xs = StandardScaler().fit(x)
    clf = GaussianProcessClassifier(
        ConstantKernel(1.0, (1e-3, 1e5)) * Matern(np.ones(5), (1e-2, 1e5), nu=2.5),
        random_state=42).fit(xs.transform(x), train_df["hit"].astype(int))
    h = train_df.loc[train_df["hit"] & np.isfinite(train_df["arrival_time_hr"])]
    yh = h[["arrival_time_hr"]].to_numpy(float); ys = StandardScaler().fit(yh)
    reg = GaussianProcessRegressor(
        ConstantKernel(1.0, (1e-3, 1e5)) * Matern(np.ones(5), (1e-2, 1e4), nu=2.5)
        + WhiteKernel(1e-5, (1e-10, 1e1)), n_restarts_optimizer=2, random_state=42
    ).fit(xs.transform(h[PARAM_NAMES].to_numpy(float)), ys.transform(yh).ravel())
    def predict_hit(theta):
        return clf.predict_proba(xs.transform(np.atleast_2d(theta)))[:, 1]
    def predict_arrival(theta):
        m, s = reg.predict(xs.transform(np.atleast_2d(theta)), return_std=True)
        return ys.inverse_transform(m.reshape(-1, 1)).ravel(), s * float(ys.scale_[0])
    return predict_hit, predict_arrival

## Your task

Build the scaling experiment. For each training size you will train a surrogate, score it on the
**fixed test set**, and record three numbers: classifier accuracy, arrival-time MAE (hours), and
the GP's own mean predictive std (hours) on the test hits.

In [ ]:
# ------------------------------ YOUR CODE HERE ------------------------------

# For each N in SIZES:
#   1. Fit a surrogate on the first N rows of `train_pool`           -> fit_surrogate(...)
#   2. Classifier skill:  predict_hit on `test[PARAM_NAMES]`, threshold at 0.5,
#      and compare to `test_hit_true` with accuracy_score              -> acc
#   3. Arrival skill:     predict_arrival on `X_test_hits` gives (mean, std);
#      mae = mean_absolute_error(y_test_hits, mean)                     -> mae
#   4. Honesty of the uncertainty:  mean of that returned std          -> mean_std
#   5. Collect (N, acc, mae, mean_std).
# Finally, plot mae vs N and mean_std vs N (two panels).


## Questions

1. Roughly where does the **MAE stop improving** - how many HUXt runs are "enough" for this event?
2. Does the GP's **mean std** end up *comparable to* the MAE (well-calibrated) or **much smaller**
   (over-confident)? Why is an over-confident surrogate dangerous for a forecast?
3. Given your curve, how many runs would you budget for a brand-new event in Exercise 4?

### Stretch
- Repeat with a *random* subset of size N instead of the first N (reshuffle `train_pool`): does the
  space-filling order of the design matter at small N?
- Add a third panel for classifier **accuracy** vs N - does the hit/miss boundary or the arrival
  time need more runs to pin down?